<a href="https://colab.research.google.com/github/AKSHAYKUMAR1505/AKSHAYKUMAR1505/blob/main/starter_ORD_no_neo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 Starter Kit: Complete the Grounding (no Neo4j version)

**Your dataset: Orders, inventory & delivery (code ORD).** An online retailer's operations log: orders, stock counts, warehouse and receiving, deliveries, returns, billing and support.

**The idea.** Retrieval finds records that *sound* similar. It cannot tell whether two records describe the *same event*. The graph can, **if it knows which parts of a record identify an event.** That knowledge is a list of regular expressions, and writing it well is your job.

**What is given and working:** retrieval, the baseline RAG, graph building (lookup tables) from a *pattern registry*, and the grounded answer step. The registry has **one pattern** (Order ID).

**What you do**
0. **Plug in your own LLM.** Add your key from any provider and, if needed, edit the one `call_llm` function so the notebook runs on it. Nothing else in the notebook touches the provider.
1. Explore the data and decide which other patterns identify a single event (**Kind 1**) and which are incidental fields reused across unrelated events (**Kind 2**: places, amounts, levels or periods).
2. Add your patterns to the registry, each with a reason. List the lookalikes you *rejected*, and why.
3. Run the public question pack against baseline and grounded. Explain every outcome.
4. Say where the baseline fails and your graph works, and **one failure you still have.**

**No database needed.** This version keeps the graph in plain Python lookup tables, already built for you, so there is no Neo4j to set up. Your work is the registry. The judging is identical to the Neo4j version.

**What you submit:** this notebook, run top to bottom on a fresh runtime. The judges read your registry, your outcomes table and your remaining-failure note, then run a hidden question pack against your graph.

> Linking on the wrong thing is as bad as linking on nothing: a pattern that links unrelated records makes the system *invent* connections. Add a pattern only if you can say why two records sharing it are the same event.

## 1. Setup

In [3]:
pip install -q sentence-transformers hnswlib openai python-dotenv

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready


### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [5]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "llama-3.1-70b-versatile"
LLM_BASE_URL = "https://api.groq.com/openai/v1"

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

LLM check: ready


### No database needed: the graph lives in lookup tables

This version of the starter holds the graph in plain Python lookup tables instead of a Neo4j database, so there is nothing to sign up for or connect. Section 7 builds the tables from your registry and `graph_check` in section 8 walks them. They answer the same question a graph database would: *is there a path of shared entities between these two records?*

## 2. The dataset: Orders, inventory & delivery

142 records in 12 categories. An online retailer's operations log: orders, stock counts, warehouse and receiving, deliveries, returns, billing and support. Four pairs of categories deliberately share vocabulary. Most records stand alone; some carry an identifier or a code.

In [6]:
DATASET_ID = "ORD"
DATASET_NAME = "Orders, inventory & delivery"

rec_categories = {
    'Order_Intake': [
        'A new online order was received and queued for payment verification',
        'Order validation flagged a mismatched shipping postcode',
        'A bulk order from a corporate buyer was split into smaller batches',
        'An order was placed on hold pending address confirmation',
        'Customer cancelled an order before it was picked',
        'A pre-order for an upcoming product was accepted into the queue',
        'Duplicate order submission was detected and merged',
        'Gift order notes were attached to the packing instruction',
        'An express order was prioritised ahead of the standard queue',
        'Order O1042 was placed for a set of kitchen shelves and queued for payment verification',
        'Order O1063 was placed on hold pending address confirmation',
        'Order O1077 was partially fulfilled and the remaining items were backordered',
        'Order O1090 was split into two parcels because the items shipped from different warehouses',
    ],
    'Inventory_Counts': [
        'Cycle count found a small variance between system stock and shelf stock',
        'Safety stock thresholds were reviewed for fast-moving items',
        'A damaged pallet was written off from available inventory',
        'Stock transfer between two warehouses was recorded in the ledger',
        'Slow-moving items were flagged for a clearance discount',
        'Inventory snapshot was frozen for the quarterly audit',
        'A mis-scanned item was corrected in the stock system',
        'Reorder points were recalculated after a demand spike',
        'Obsolete stock was moved to the disposal list',
        'Order O1077 triggered a stock shortage alert for a backordered item',
        'Order O1077 reserved 500 units of printer paper from the main warehouse',
        'A stock transfer of 500 units of cleaning supplies was recorded between two warehouses',
        'A write-off of 240 units of expired snacks was approved after the audit',
        'A cycle count in Zone B found one pallet missing',
        'Return RMA-4471 was inspected and the blender was written off as scrap',
    ],
    'Warehouse_Storage': [
        'Pallet racking in the cold storage area was inspected for damage',
        'Slotting was optimised to place popular items near the packing stations',
        'A forklift battery was swapped at the charging bay',
        'Overflow stock was moved to the temporary storage yard',
        'Fire safety inspection was completed for the main storage hall',
        'Shelving labels were reprinted after a re-slotting exercise',
        'Temperature log for the cold storage was reviewed by the supervisor',
        'A mezzanine floor was cleared for seasonal storage',
        'Pest control treatment was scheduled for the dry goods section',
        'Zone B was reorganised to hold fast-moving products',
        'A forklift collision at Dock 4 was reported to the safety officer',
        'Driver D15 reported a damaged ramp at Dock 9 to the warehouse supervisor',
    ],
    'Inbound_Receiving': [
        'A supplier truck arrived at the gate for unloading',
        'Received goods were checked against the advance shipping notice',
        'A short shipment was reported by the receiving team',
        'Damaged cartons were photographed and logged at receipt',
        'Putaway of received pallets was completed before shift end',
        'Receiving team quarantined a delivery with a missing label',
        'An unscheduled supplier delivery was turned away at the gate',
        'Inbound appointment slots were rebooked after a traffic delay',
        'Receiving staff counted a mixed pallet carton by carton',
        'Dock 4 was reserved for an incoming supplier truck this morning',
        'A shipment of 240 units of phone cases arrived from a supplier',
        'Driver D12 waited ninety minutes at Dock 9 for a free loading slot',
    ],
    'Outbound_Delivery': [
        'A delivery was attempted but the recipient was not at home',
        'Same-day delivery slots filled up before noon',
        'A parcel was rerouted after the courier hub flagged a wrong address',
        'Delivery confirmation photo was uploaded by the driver',
        'A late delivery was escalated to the regional delivery manager',
        'Out-for-delivery notification was sent to the customer',
        'Last-mile delivery to a remote area was rescheduled for the next day',
        'A signature-required parcel was left with a neighbour by mistake',
        'Delivery route was re-sequenced after a road closure',
        'Order O1042 was delivered to the customer and signed for at the door',
        'Order O1051 was marked out for delivery by the courier',
        'Driver D07 completed the morning route with every parcel delivered',
        'Driver D12 reported a blocked access road on the evening route',
        "Customer C219's parcel was rerouted to the new address by the courier",
    ],
    'Returns_RMA': [
        'A customer requested a return for a defective item',
        'Returned goods were inspected and restocked as new',
        'A return was rejected because the return window had passed',
        'Returns team issued a prepaid return label to a customer',
        'A refurbished item was graded and moved to the outlet stock',
        'Return pickup was scheduled with the courier',
        'A wrong-item return was approved without inspection',
        'Returns processing backlog was cleared after the weekend',
        'A return reason survey was added to the return form',
        'Return RMA-4471 was opened for a defective blender and a prepaid label was issued',
        'Return RMA-3092 was approved for a wrong-colour item',
        'Customer C233 returned a pair of shoes within the return window',
    ],
    'Billing_Refunds': [
        'A refund was issued to the original payment method',
        "Billing corrected a duplicate charge on a customer's card",
        'Refund processing time was extended due to a bank holiday',
        'A partial refund was approved for a late delivery',
        'Invoice was regenerated after a billing address change',
        'A chargeback notice was received from a card issuer',
        'Store credit was issued in place of a cash refund',
        "Billing reconciled the day's card settlements",
        'A promotional discount was reversed after an order edit',
        'Customer C204 received a partial refund for the damaged parcel',
        'Order O1088 was charged a re-delivery fee after two failed delivery attempts',
    ],
    'Supplier_Payments': [
        'Supplier invoice was approved for payment after three-way matching',
        'A payment run was scheduled for the end of the month',
        'Supplier requested early payment in exchange for a discount',
        'Procurement disputed a supplier invoice for incorrect pricing',
        'A supplier payment was held pending a quality complaint',
        'Bank details of a supplier were verified before the next payment',
        'Credit note from a supplier was applied against an open invoice',
        'Accounts payable aged balances were reviewed by finance',
        'Freight charges were added to the supplier invoice',
    ],
    'Customer_Support': [
        'A customer asked for an update on a delayed parcel',
        'Support agent resolved a login problem on the order tracking page',
        'A complaint about damaged packaging was escalated to quality',
        'Chat support handled a question about delivery charges',
        'A customer asked to change the delivery address after dispatch',
        'Support team sent an apology voucher for a missed delivery slot',
        'A customer enquired about the return policy for sale items',
        'Support tickets were tagged by topic for the weekly review',
        'A customer praised the quick delivery in a feedback form',
        'Customer C204 complained that a parcel arrived with damaged packaging',
        'Customer C219 asked to change the delivery address after dispatch',
        'Order O1063 was the subject of a customer request to change the delivery address',
    ],
    'Carrier_Management': [
        'Carrier performance scorecard was shared with the logistics team',
        'A new regional courier was onboarded for last-mile coverage',
        'Carrier rates were renegotiated for heavy parcels',
        'A courier contract renewal was sent for legal review',
        'Carrier pickup time was moved earlier for the evening dispatch',
        'Customs paperwork was prepared for an international shipment',
        'A carrier strike caused a backlog at the sorting hub',
        'Fuel surcharge changes were communicated by the carrier',
        'Carrier tracking feed was fixed after an integration error',
        'Driver D07 was commended by the regional manager for on-time performance',
        'Driver D15 completed the new-joiner safety briefing',
        'Carrier surcharges were announced for the holiday season',
    ],
    'Packaging': [
        'Packing station was restocked with bubble wrap and tape',
        'A new eco-friendly mailer bag was approved for light items',
        'Fragile items were double-boxed before dispatch',
        'Packing slips were reprinted after a printer jam',
        'Oversized items were wrapped in corrugated board',
        'Packers were trained on the new gift wrapping process',
        'A packing error was caught at the final scan',
        'Cartons of the wrong size were returned to the supplier',
        'Shrink wrap was used to secure a pallet for dispatch',
        'Order O1051 was double-boxed because it contained a glass lamp',
        'Extra temporary packers were hired for the holiday season',
    ],
    'Demand_Planning': [
        'Forecast was revised upward after a successful marketing campaign',
        'Planners reviewed the promotional calendar for the next quarter',
        'Demand for winter wear was projected to rise in the coming weeks',
        'Planning team agreed on a buffer stock for a new launch',
        'A forecast accuracy report was shared with the category managers',
        "Seasonal demand curves were updated with last year's data",
        'Planners flagged a supplier lead time increase',
        'A product was discontinued from the forecast model',
        'Weekly sales and operations meeting reviewed capacity constraints',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

Dataset ORD: 142 records, 12 categories


## 3. Retrieval (given: this part works)

In [7]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Index built: 142 records


## 4. Baseline RAG (given: no grounding)

The plain retrieval-and-generation pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [8]:
# Baseline RAG prompt -- no graph grounding. temperature=0 reduces variation.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. Explore: what could identify one event?

Read the records below. For each string with digits, ask: *if I saw this same string in a different record, would it still point at the same event?*

In [9]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(rec_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

R9   Order O1042 was placed for a set of kitchen shelves and queued for payment verification
R10  Order O1063 was placed on hold pending address confirmation
R11  Order O1077 was partially fulfilled and the remaining items were backordered
R12  Order O1090 was split into two parcels because the items shipped from different warehouses
R22  Order O1077 triggered a stock shortage alert for a backordered item
R23  Order O1077 reserved 500 units of printer paper from the main warehouse
R24  A stock transfer of 500 units of cleaning supplies was recorded between two warehouses
R25  A write-off of 240 units of expired snacks was approved after the audit
R27  Return RMA-4471 was inspected and the blender was written off as scrap
R38  A forklift collision at Dock 4 was reported to the safety officer
R39  Driver D15 reported a damaged ramp at Dock 9 to the warehouse supervisor
R49  Dock 4 was reserved for an incoming supplier truck this morning
R50  A shipment of 240 units of phone cases arrived

## 6. YOUR TASK: the pattern registry

Each entry becomes a graph node label. Two records that mention the *same identifier* get connected through that node. Edit this cell, then run it.

In [24]:

import re

# ============================================================
# SECTION 6: IDENTIFIER PATTERN REGISTRY
# Orders, Inventory and Delivery (ORD)
# ============================================================

PATTERNS = [
    {
        "name": "Order",
        "regex": r"\bOrder\s+(O\d{4})\b",
        "kind": 1,
        "example": "Order O1042",
        "why": (
            "An order number identifies a specific order. "
            "Records referencing the same order ID can be "
            "connected as references to that order."
        ),
    },
    {
        "name": "Return_ID",
        "regex": r"\b(?:Return\s+)?(RMA-\d+)\b",
        "kind": 1,
        "example": "Return RMA-4471",
        "why": (
            "An RMA identifier identifies a specific return. "
            "Records sharing the same RMA identifier can be "
            "linked as references to that return."
        ),
    },
]

# Candidates deliberately rejected as automatic event-level links.
REJECTED_CANDIDATES = [
    {
        "candidate": "Customer IDs such as C204 and C219",
        "kind": 2,
        "why_rejected": (
            "A customer can have multiple unrelated orders, "
            "deliveries, refunds, and support interactions. "
            "A shared customer ID alone does not establish "
            "that two records describe the same event."
        ),
    },
    {
        "candidate": "Warehouse locations such as Dock 4 and Dock 9",
        "kind": 2,
        "why_rejected": (
            "A dock can be used for multiple unrelated events. "
            "Sharing a location does not prove that records "
            "refer to the same shipment or operation."
        ),
    },
    {
        "candidate": "Driver IDs such as D07, D12, and D15",
        "kind": 2,
        "why_rejected": (
            "A driver can participate in multiple deliveries "
            "and warehouse activities. A shared driver ID "
            "does not prove that two records describe the "
            "same delivery event."
        ),
    },
    {
        "candidate": "Repeated quantities such as 240 or 500 units",
        "kind": 2,
        "why_rejected": (
            "The same quantity can occur in unrelated stock "
            "counts, receipts, transfers, and write-offs. "
            "A quantity is not a unique event identifier."
        ),
    },

]


# ============================================================
# REGISTRY VALIDATION
# ============================================================

def validate_registry():
    """Check required fields, regex syntax and capture groups."""
    problems = []

    for pattern in PATTERNS:
        name = pattern.get("name", "?")

        for field in ("name", "regex", "kind", "example", "why"):
            if pattern.get(field) in (None, ""):
                problems.append(f"{name}: missing '{field}'")

        if pattern.get("name") and not re.fullmatch(
            r"[A-Za-z][A-Za-z0-9_]*", pattern["name"]
        ):
            problems.append(
                f"{name}: name must contain only letters, "
                "digits and underscores, and start with a letter"
            )

        try:
            regex = re.compile(pattern.get("regex", ""))

            if regex.groups != 1:
                problems.append(
                    f"{name}: regex must contain exactly "
                    "one capture group around the identifier"
                )

        except re.error as error:
            problems.append(
                f"{name}: invalid regular expression ({error})"
            )

        if pattern.get("kind") not in (1, 2):
            problems.append(f"{name}: kind must be 1 or 2")

    return problems


# ============================================================
# DISPLAY THE REGISTRY
# ============================================================

def show_registry():
    print("\nIDENTIFIER PATTERN REGISTRY")
    print("-" * 75)

    for pattern in PATTERNS:
        print(f"\nPattern: {pattern['name']}")
        print(f"Kind:    {pattern['kind']}")
        print(f"Regex:   {pattern['regex']}")
        print(f"Example: {pattern['example']}")
        print(f"Why:     {pattern['why']}")

    print("\nREJECTED CANDIDATES")
    print("-" * 75)

    for candidate in REJECTED_CANDIDATES:
        print(f"\nCandidate: {candidate['candidate']}")
        print(f"Kind:      {candidate['kind']}")
        print(f"Reason:    {candidate['why_rejected']}")


# ============================================================
# RUN VALIDATION
# ============================================================

problems = validate_registry()

print(
    "Registry problems:",
    problems if problems else "none"
)

show_registry()


Registry problems: none

IDENTIFIER PATTERN REGISTRY
---------------------------------------------------------------------------

Pattern: Order
Kind:    1
Regex:   \bOrder\s+(O\d{4})\b
Example: Order O1042
Why:     An order number identifies a specific order. Records referencing the same order ID can be connected as references to that order.

Pattern: Return_ID
Kind:    1
Regex:   \b(?:Return\s+)?(RMA-\d+)\b
Example: Return RMA-4471
Why:     An RMA identifier identifies a specific return. Records sharing the same RMA identifier can be linked as references to that return.

REJECTED CANDIDATES
---------------------------------------------------------------------------

Candidate: Customer IDs such as C204 and C219
Kind:      2
Reason:    A customer can have multiple unrelated orders, deliveries, refunds, and support interactions. A shared customer ID alone does not establish that two records describe the same event.

Candidate: Warehouse locations such as Dock 4 and Dock 9
Kind:      2


## 7. Build the lookup tables from your registry (given)

Re-run this cell every time you change the registry. Three tables hold the graph: `mentions` (one row per record-entity match) and the same rows indexed both ways (`entities_of`, `records_of`). The summary shows how many links each pattern created: a pattern with **0 shared identifiers** adds nothing, and one that links *many* records deserves suspicion.

In [11]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

# Table 1: one row per mention: (record_id, entity_type, entity_value)
mentions = []
for i, text in enumerate(rec_texts):
    for etype, value in extract_entities(text):
        mentions.append((f"R{i}", etype, value))

# Tables 2 and 3: the same rows indexed both ways, so we can walk record -> entity -> record
entities_of = {}   # "R12" -> {("Student", "S001"), ...}
records_of = {}    # ("Student", "S001") -> {"R12", "R57", ...}
for rec, etype, value in mentions:
    entities_of.setdefault(rec, set()).add((etype, value))
    records_of.setdefault((etype, value), set()).add(rec)

print(f"Lookup tables built from your registry: {len(mentions)} mention rows")
for name in sorted({m[1] for m in mentions}):
    ids = {}
    for rec, etype, value in mentions:
        if etype == name:
            ids[value] = ids.get(value, 0) + 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {sum(ids.values()):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

Lookup tables built from your registry: 14 mention rows
  Order       11 mentions |   6 distinct identifiers | 4 shared by 2+ records (= links)
  Return_ID    3 mentions |   2 distinct identifiers | 1 shared by 2+ records (= links)


In [12]:
# What the lookup table looks like: the first 12 rows
print(f"{'record':<8}{'entity_type':<14}entity_value")
for row in mentions[:12]:
    print(f"{row[0]:<8}{row[1]:<14}{row[2]}")

record  entity_type   entity_value
R9      Order         O1042
R10     Order         O1063
R11     Order         O1077
R12     Order         O1090
R22     Order         O1077
R23     Order         O1077
R27     Return_ID     RMA-4471
R61     Order         O1042
R62     Order         O1051
R75     Return_ID     RMA-4471
R76     Return_ID     RMA-3092
R88     Order         O1088


## 8. Grounded answer (given)

`graph_check` walks the lookup tables to see whether two records are connected (and through which entity). The grounded step asks it about each pair of retrieved records, then hands the LLM those facts with record IDs. `temperature=0` is used to reduce variation between runs.

In [13]:
from collections import deque

def graph_check(id_a, id_b, max_hops=6):
    """Are two records connected? Returns {"disconnected": bool, "via": [...]}.
    Walks record -> entity -> record using the lookup tables; each step is one hop,
    and the walk stops after 6 hops."""
    start, goal = ("R", id_a), ("R", id_b)
    prev, queue = {start: None}, deque([(start, 0)])
    while queue:
        node, depth = queue.popleft()
        if node == goal:
            break
        if depth >= max_hops:
            continue
        if node[0] == "R":
            neighbours = [("E", etype, value) for etype, value in entities_of.get(node[1], ())]
        else:
            neighbours = [("R", rec) for rec in records_of[(node[1], node[2])]]
        for nb in neighbours:
            if nb not in prev:
                prev[nb] = node
                queue.append((nb, depth + 1))
    if goal not in prev:
        return {"disconnected": True, "via": None}
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = prev[node]
    return {"disconnected": False, "via": [f"{n[1]} {n[2]}" for n in reversed(path) if n[0] == "E"]}


GROUNDED_PROMPT = """You answer questions about online retail fulfilment (orders, stock, warehouse, couriers, returns) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Quantities, dock numbers, zone letters, seasons, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 9. The public question pack

8 adversarial questions (the right answer is *no recorded connection*) and 3 positive controls (the right answer is *yes*). The judges will run a **different, hidden** pack of the same shapes, so fix your **registry**, not individual questions.

In [14]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': "Was the refund issued to the customer's card paid out of the supplier payment run "
              'scheduled for month end?',
  'trap_records': ['A refund was issued to the original payment method',
                   'A payment run was scheduled for the end of the month'],
  'correct_answer_must_state': 'No stated connection: a customer refund and a supplier payment run '
                               'are separate records, both about money.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (money)'},
 {'id': 'P2',
  'question': 'Was the overflow stock moved to the temporary yard because the supplier truck '
              'arrived at the gate?',
  'trap_records': ['Overflow stock was moved to the temporary storage yard',
                   'A supplier truck arrived at the gate for unloading'],
  'correct_answer_must_state': 'No stated connection: a storage move and a truck arrival share no '
                               'order, shipment or named entity.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (place)'},
 {'id': 'P3',
  'question': 'Was the parcel rerouted because of the carrier strike at the sorting hub?',
  'trap_records': ['A parcel was rerouted after the courier hub flagged a wrong address',
                   'A carrier strike caused a backlog at the sorting hub'],
  'correct_answer_must_state': 'No stated connection: the reroute cites a wrong address, and '
                               'nothing ties it to the strike.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (movement)'},
 {'id': 'P4',
  'question': "Was the customer's defective-item return the same case as the damaged packaging "
              'complaint?',
  'trap_records': ['A customer requested a return for a defective item',
                   'A complaint about damaged packaging was escalated to quality'],
  'correct_answer_must_state': 'No evidence these two records name the same customer or order; '
                               'they are separate records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (complaint)'},
 {'id': 'P5',
  'question': 'Did the incoming supplier truck cause the forklift collision at Dock 4?',
  'trap_records': ['Dock 4 was reserved for an incoming supplier truck this morning',
                   'A forklift collision at Dock 4 was reported to the safety officer'],
  'correct_answer_must_state': 'No stated connection: both mention Dock 4, but a dock is a place '
                               'reused by many trucks, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: dock'},
 {'id': 'P6',
  'question': 'Was the pallet missing in the cycle count lost during the Zone B reorganisation?',
  'trap_records': ['Zone B was reorganised to hold fast-moving products',
                   'A cycle count in Zone B found one pallet missing'],
  'correct_answer_must_state': 'No stated connection: sharing a zone name is not sharing an event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: zone'},
 {'id': 'P7',
  'question': 'Were the 240 units of phone cases the same units that were written off?',
  'trap_records': ['A shipment of 240 units of phone cases arrived from a supplier',
                   'A write-off of 240 units of expired snacks was approved after the audit'],
  'correct_answer_must_state': 'No stated connection: the quantities match by coincidence, and the '
                               'products differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: quantity'},
 {'id': 'P8',
  'question': 'Was the forklift battery swapped because the forecast was revised upward?',
  'trap_records': ['A forklift battery was swapped at the charging bay',
                   'Forecast was revised upward after a successful marketing campaign'],
  'correct_answer_must_state': 'No stated connection between a battery swap and a forecast '
                               'revision; any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Order O1051 double-boxed and also marked out for delivery?',
  'shared_identifier': 'Order O1051',
  'shared_records': ['Order O1051 was double-boxed because it contained a glass lamp',
                     'Order O1051 was marked out for delivery by the courier'],
  'correct_answer_must_state': 'Yes -- both records name Order O1051.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Was Order O1063 put on hold, and did a customer later ask to change its delivery '
              'address?',
  'shared_identifier': 'Order O1063',
  'shared_records': ['Order O1063 was placed on hold pending address confirmation',
                     'Order O1063 was the subject of a customer request to change the delivery '
                     'address'],
  'correct_answer_must_state': 'Yes -- both records name Order O1063.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Did Customer C204 both report damaged packaging and receive a partial refund?',
  'shared_identifier': 'Customer C204',
  'shared_records': ['Customer C204 complained that a parcel arrived with damaged packaging',
                     'Customer C204 received a partial refund for the damaged parcel'],
  'correct_answer_must_state': 'Yes -- both records name Customer C204.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

Public pack: 8 adversarial + 3 positive-control questions


In [15]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### Adversarial questions

In [16]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)

[P1] Was the refund issued to the customer's card paid out of the supplier payment run scheduled for month end?
   Baseline : No. The refund was processed separately and sent back to the customer’s original payment method, not taken out of the supplier payment run that was scheduled for month‑end.
   Grounded : No recorded link exists between the refund issued to the customer's card (R78) and the supplier payment run scheduled for month‑end (R90).
   Graph    : NOT connected | expected NOT connected -> PASS
   Must say : No stated connection: a customer refund and a supplier payment run are separate records, both about money.

[P2] Was the overflow stock moved to the temporary yard because the supplier truck arrived at the gate?
   Baseline : No – the context does not indicate that the overflow stock was moved to the temporary yard because the supplier truck arrived at the gate.
   Grounded : No recorded link exists between the overflow‑stock movement (R31) and the supplier truck arriv

### Positive controls: can your agent say *yes*?

In [17]:
pos_rows = run_pack(PUBLIC_POSITIVE)

[PP1] Was Order O1051 double-boxed and also marked out for delivery?
   Baseline : Yes. Order O1051 was both double‑boxed and marked out for delivery.
   Grounded : Yes. Order O1051 was both double‑boxed (record [R131]) and marked out for delivery (record [R62]).
   Graph    : CONNECTED via Order O1051 | expected connected -> PASS
   Must say : Yes -- both records name Order O1051.

[PP2] Was Order O1063 put on hold, and did a customer later ask to change its delivery address?
   Baseline : Yes – Order O1063 was placed on hold pending address confirmation, and a customer subsequently requested that its delivery address be changed.
   Grounded : Yes.   - Order O1063 was placed on hold pending address confirmation【R10】.   - A customer later requested to change the delivery address for the same order【R109】.    The two records are linked through Order O1063.
   Graph    : CONNECTED via Order O1063 | expected connected -> PASS
   Must say : Yes -- both records name Order O1063.

[PP3] Did C

## 10. YOUR TASK: explain the outcomes

Fill in one line of `OUTCOMES` for each of the 11 public questions (P1-P8 and PP1-PP3). Each line has three fields. Read the **Baseline**, **Grounded**, **Graph** and **Must say** lines that the pack printed for that question, then decide:

| Field | The question it answers | How to decide |
| --- | --- | --- |
| `baseline_fabricated` | Did the plain model (no graph) **invent** something the records do not support? | Compare the **Baseline** answer with the **Must say** line. `True` only if the baseline claimed a link, a cause or "same person" that the records do not show. If it said "no", said "cannot tell", or said "yes" to a link that really exists, it did **not** fabricate: write `False`. |
| `grounded_correct` | Was **your** grounded answer right? | Read the **Grounded** answer and compare it with **Must say**. Use `True` only when the answer is correct. The **Graph** PASS/FAIL line checks the record-pair connection; it does not by itself verify the LLM wording. |
| `explanation` | Why did it come out that way? | One sentence in terms of **your patterns**: which pattern matched (or did not), and which word was the shared ID, or a place, level, amount or period that is *not* an ID. |

**Most baselines are right on most questions.** `False` for `baseline_fabricated` is a normal, honest result; a long list of `True` that you did not actually see in the answers will lose marks. We score accurate, specific explanations, not the number of failures you found.

**How to write the explanation**

- Linked question (PP): *"Both records name <the ID>, and our <Name> pattern extracts it, so they are connected."*
- Not linked (P), shared word is a place, level, amount or period: *"Both mention <the word>, but that is a <place / amount / period>, a Kind 2 field reused by many events, not an event ID. We do not link on it."*
- Not linked (P), nothing shared: *"The two records name no shared ID, so no pattern links them."*
- Do not write "it might have..." or "it is not specified". That is guessing, and guessing is what the baseline does.

Example of the shape (not one of your questions):

```python
{"id": "P0", "baseline_fabricated": False, "grounded_correct": True,
 "explanation": "Both mention Shelf 3, but a shelf is a place, not an event ID, so no link."},
```

Then name one failure you still have in `REMAINING_FAILURE`. Be specific: a short honest note beats a vague claim.

In [18]:
OUTCOMES = [
    # One dictionary per question, in this shape (see the table above):
    # {"id": "P1", "baseline_fabricated": ..., "grounded_correct": ..., "explanation": "..."},
    # Fill True or False for the first two fields; do not leave None.
    # Cover P1-P8 and PP1-PP3.
]

REMAINING_FAILURE = """
TODO: one case, or kind of case, where your system still gets it wrong (or could), and why.
"""


## 11. Submission summary

Run this last. This is what the judges read first.

In [20]:

# Compatibility fix for the submission summary
def show_register():
    return show_registry()


In [21]:
import json

def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    # one machine-readable line the judges' tool reads: keep it
    print("REGISTRY_JSON:", json.dumps([{"name": p["name"], "regex": p["regex"]} for p in PATTERNS]))
    print("DATASET_ID:", DATASET_ID)
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()

=== PATTERN REGISTER ===

IDENTIFIER PATTERN REGISTRY
---------------------------------------------------------------------------

Pattern: Order
Kind:    1
Regex:   \bOrder\s+(O\d{4})\b
Example: Order O1042
Why:     An order number identifies a specific order. Records referencing the same order ID can be connected as references to that order.

Pattern: Return_ID
Kind:    1
Regex:   \b(?:Return\s+)?(RMA-\d+)\b
Example: Return RMA-4471
Why:     An RMA identifier identifies a specific return. Records sharing the same RMA identifier can be linked as references to that return.

REJECTED CANDIDATES
---------------------------------------------------------------------------

Candidate: Customer IDs such as C204 and C219
Kind:      2
Reason:    A customer can have multiple unrelated orders, deliveries, refunds, and support interactions. A shared customer ID alone does not establish that two records describe the same event.

Candidate: Warehouse locations such as Dock 4 and Dock 9
Kind:      2

## 12. Before you submit

- Restart the runtime and **Run all**: it must work from a fresh state.
- `validate_registry()` reports no problems.
- `OUTCOMES` has an entry for every public question; `REMAINING_FAILURE` is filled in.
